## Importar librerías y cargar API key de Gemini

In [ ]:
!pip install -q -U google-generativeai

In [1]:
# Importación de librerías necesarias
from google import genai
from google.genai import types
from google.colab import userdata
import pandas as pd
import re
from tqdm import tqdm
tqdm.pandas()
import os

# Cargar API key de Gemini desde variables seguras de Colab
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
# Inicializar cliente de Gemini
client = genai.Client(api_key=GEMINI_API_KEY)
# Seleccionar modelo
MODEL = "gemini-3.1-pro-preview"
# Límite diario de la API (dejamos 10 de margen sobre el límite real de 250)
RPD_LIMIT = 240

## Cargar dataset

In [2]:
# conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Cargar datasets individuales por tarea
path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/LLM/"
path_tara5 = path + "tara5/"

df_anx = pd.read_csv(path + "anxiety_llm.csv", sep=",")
df_dep = pd.read_csv(path + "depression_llm.csv", sep=",")
df_ed = pd.read_csv(path + "ed_llm.csv", sep=",")
df_multi = pd.read_csv(path + "multiclass_llm.csv", sep=",")

df_anx_tara5 = pd.read_csv(path_tara5 + "anxiety_llm_tara5.csv")
df_dep_tara5 = pd.read_csv(path_tara5 + "depression_llm_tara5.csv")
df_ed_tara5  = pd.read_csv(path_tara5 + "ed_llm_tara5.csv")
df_multi_tara5 = pd.read_csv(path_tara5 + "multiclass_llm_tara5.csv")

# Seleccionar tarea de clasificacion: anx | dep | ed | multi
task = "anx" # <--- Cambiar aquí

# Asignar el dataframe correspondiente a la tarea seleccionada
if task == "anx":
  df = df_anx
  df_tara5 = df_anx_tara5
elif task == "dep":
  df = df_dep
  df_tara5 = df_dep_tara5
elif task == "ed":
  df = df_ed
  df_tara5 = df_ed_tara5
elif task == "multi":
  df = df_multi
  df_tara5 = df_multi_tara5

## Función para construir el prompt

Se añade el texto a clasificar a un prompt fijo que cambio según la tarea.

In [4]:
def build_prompt(text, task):
  """ Construcción dinámica del prompt según tarea.
  Devuelve el texto completo que se enviará al modelo.
  """

  if task == "anx":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de ansiedad.

La ansiedad se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa. Estas experiencias suelen generar malestar emocional sostenido y dificultad para relajarse.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de ansiedad
0: el texto no muestra indicios de ansiedad

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "dep":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de depresión.

La depresión se asocia a un estado persistente de tristeza, apatía o vacío emocional, acompañado de una disminución del interés o la capacidad de disfrute en actividades habituales. Estas experiencias pueden incluir pensamientos negativos recurrentes, fatiga emocional y una percepción pesimista de uno mismo o del futuro.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de depresión
0: el texto no muestra indicios de depresión

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "ed":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de un trastorno de la conducta alimentaria.

Los trastornos de la conducta alimentaria se asocian a una preocupación persistente por el peso corporal, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida, el cuerpo o el control del peso. Estas preocupaciones suelen generar malestar emocional, culpa, ansiedad en torno a la alimentación y conductas de evitación o control.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de un trastorno de la conducta alimentaria
0: el texto no muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  else:
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede reflejar indicios de uno de los siguientes estados psicológicos, o no reflejar ninguno de ellos.

Ansiedad: se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa, generando malestar emocional y dificultad para relajarse.

Depresión: se asocia a un estado persistente de tristeza, apatía o vacío emocional, junto con una disminución del interés o la capacidad de disfrute en actividades habituales, pensamientos negativos recurrentes y una visión pesimista de uno mismo o del futuro.

Trastornos de la conducta alimentaria: se asocian a una preocupación persistente por el peso, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida o el cuerpo, así como malestar emocional, culpa o ansiedad en torno a la alimentación.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

0: el texto no muestra indicios de ninguno de los estados descritos
1: el texto muestra indicios de ansiedad
2: el texto muestra indicios de depresión
3: el texto muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""

## Calcular número de tokens y coste

In [ ]:
def total_tokens_dataset(df, task):
  """ Calcula el número total de tokens de entrada
  para un dataset completo, usando el tokenizador del modelo
  """
  total_tokens = 0

  for text in df["texto"]:
    # Contar tokens del prompt completo (instrucciones + texto)
    response = client.models.count_tokens(
        model=MODEL,
        contents=build_prompt(text, task),
    )
    total_tokens += response.total_tokens

  return total_tokens


# Cálculo de tokens por dataset
total_anx = total_tokens_dataset(df_anx, "anx")
print("DATASET ANSIEDAD")
print("Tokens:", total_anx)
print("Media por texto:", total_anx / len(df_anx))

total_dep = total_tokens_dataset(df_dep, "dep")
print("DATASET DEPRESIÓN")
print("Tokens:", total_dep)
print("Media por texto:", total_dep / len(df_dep))

total_ed = total_tokens_dataset(df_ed, "ed")
print("DATASET ED")
print("Tokens:", total_ed)
print("Media por texto:", total_ed / len(df_ed))

total_multi = total_tokens_dataset(df_multi, "multi")
print("DATASET MULTICLASE")
print("Tokens:", total_multi)
print("Media por texto:", total_multi / len(df_multi))

# Suma total de tokens necesarios para todo el experimento
print(f"Total de tokens: {total_anx + total_dep + total_ed + total_multi}")

DATASET ANSIEDAD
Tokens: 445813
Media por texto: 891.626
DATASET DEPRESIÓN
Tokens: 378367
Media por texto: 758.2505010020041
DATASET ED
Tokens: 260571
Media por texto: 777.823880597015
DATASET MULTICLASE
Tokens: 1273296
Media por texto: 954.4947526236881
Total de tokens: 2358047


## Clasificación zero-shot con Gemini

In [5]:
def classify_text(text, task):
  """ Función de clasificación usando Gemini
  Devuelve:
  - int (0,1,2,3) si hay predicción válida
  - None si el modelo bloquea o no devuelve texto
  """

  response = client.models.generate_content(
    model=MODEL,
    contents=build_prompt(text, task),
    config={
        "temperature": 0,       # Generación determinista
        "max_output_tokens": 5, # Limitar longitud de salida
    }
  )

  # Devuelve None si no hay texto
  # (suele ocurrir porque bloquea el prompt por PROHIBITED_CONTENT)
  if not response.text:
    return None

  # Extraer número de clase mediante expresión regular
  match = re.search(r"[0-3]", response.text)

  return int(match.group()) if match else None

Nota: la API de Gemini 3.1 Pro (preview) tiene un límite de 250 RPD (requests per day) que no se amplía aunque se tenga facturación activada, al tratarse de un modelo en preview. Por ello, la clasificación se realiza en sesiones diarias de hasta RPD_LIMIT llamadas, guardando un checkpoint en cada ejecución para poder reanudar al día siguiente sin perder progreso.

In [10]:
import time

checkpoint_dir = "/content/drive/MyDrive/tfg/corpusMentalRiskES/checkpoints/"
output_dir = "/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

def classify_with_daily_limit(df, task, rpd_limit=RPD_LIMIT, run_id=None):
  """
  Clasifica el dataset respetando el límite diario de la API.

  Lógica:
  - Carga el checkpoint si existe (reanuda desde donde se quedó)
  - Procesa hasta rpd_limit textos por ejecución
  - Guarda el checkpoint al finalizar la sesión del día
  - Cuando todos los textos están clasificados, guarda el resultado final

  Parámetros:
  - df: DataFrame con columna 'texto'
  - task: tarea ('anx', 'dep', 'ed', 'multi')
  - rpd_limit: máximo de llamadas a la API en esta ejecución
  """

  if run_id is None:
    checkpoint_path = checkpoint_dir + f"checkpoint_{MODEL.replace('/', '_')}_{task}.csv"
  else:
    checkpoint_path = checkpoint_dir + f"tara5_{MODEL.replace('/', '_')}_{task}_run{run_id}.csv"

  # --- Cargar checkpoint si existe ---
  if os.path.exists(checkpoint_path):
    df_ckpt = pd.read_csv(checkpoint_path)
    df["pred"] = df_ckpt["pred"]
    ya_procesados = int(df["pred"].notna().sum())
    print(f"✅ Checkpoint encontrado: {ya_procesados}/{len(df)} textos ya procesados.")
  else:
    df["pred"] = None
    ya_procesados = 0
    print(f"🆕 Sin checkpoint previo. Empezando desde cero.")

  pendientes = int(df["pred"].isna().sum())

  if pendientes == 0:
    print("🎉 Dataset ya completado. No hay nada que procesar.")
    return df

  llamadas_hoy = 0
  bloqueados = 0

  print(f"📋 Textos pendientes: {pendientes} | Límite de hoy: {rpd_limit}")
  print(f"📅 Días restantes estimados: {-(-pendientes // rpd_limit)}")  # ceil division

  # Barra de progreso sobre los textos que procesaremos hoy
  pbar = tqdm(total=min(pendientes, rpd_limit), desc="Progreso de hoy")

  for i, row in df.iterrows():

    # Parar si alcanzamos el límite diario
    if llamadas_hoy >= rpd_limit:
      print(f"\n🛑 Límite diario alcanzado ({rpd_limit} llamadas). Vuelve mañana.")
      break

    # Saltar filas ya procesadas
    if pd.notna(df.at[i, "pred"]):
      continue

    start_time = time.time()

    df.at[i, "pred"] = classify_text(row["texto"], task)
    llamadas_hoy += 1

    if df.at[i, "pred"] is None:
      bloqueados += 1

    pbar.update(1)
    pbar.set_postfix({"bloqueados": bloqueados, "hoy": llamadas_hoy})

    # Control de rate limit (25 RPM)
    elapsed = time.time() - start_time
    sleep_time = max(0, 2.5 - elapsed)
    time.sleep(sleep_time)

  pbar.close()

  # --- Guardar checkpoint ---
  df.to_csv(checkpoint_path, index=False)
  total_procesados = int(df["pred"].notna().sum())
  print(f"\n💾 Checkpoint guardado. Total procesados: {total_procesados}/{len(df)}")

  # --- Si el dataset está completo, guardar resultado final ---
  if df["pred"].isna().sum() == 0:
    output_path = output_dir + f"{MODEL.replace('/', '_')}_{task}.csv"
    df.to_csv(output_path, index=False)
    print(f"🎉 Dataset completado. Resultado final guardado en:\n   {output_path}")

  return df, llamadas_hoy

In [ ]:
# Aplicar clasificación al dataset seleccionado
# Se almacena la predicción en una nueva columna "pred"
print(f"Iniciando sesión | Tarea: {task} | Modelo: {MODEL}")
df, _ = classify_with_daily_limit(df, task)

In [ ]:
df.to_csv(checkpoint_dir + f"checkpoint_{MODEL.replace('/', '_')}_{task}.csv", index=False)
print("Checkpoint guardado manualmente")

Checkpoint guardado manualmente


### Comprobación de predicciones

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# Reemplazar None por una clase inválida para que cuente como error
pred_labels = df["pred"].fillna(-1).astype(int)
mask = pred_labels != -1  # Excluir bloqueos

# Si es clasificación binaria
if task in ["anx", "dep", "ed"]:
    # Columna con la etiqueta real
    true_labels = df["bs"]
    f1 = f1_score(true_labels[mask], pred_labels[mask], average="binary")
else:
    # Clasificación multiclase
    true_labels = df["label_mc"]
    f1 = f1_score(true_labels[mask], pred_labels[mask], average="macro")

acc = accuracy_score(true_labels[mask], pred_labels[mask])

print("F1-score:", round(f1, 4))
print("Acc:", round(acc, 4))

F1-score: 0.9581
Acc: 0.9257


In [ ]:
# Número de bloqueos
num_blocked = df["pred"].isna().sum()
block_rate = num_blocked / len(df)

print("Bloqueos:", num_blocked)
print("Tasa de bloqueo:", round(block_rate, 4))

Bloqueos: 2
Tasa de bloqueo: 0.004


### Almacenar predicciones

In [ ]:
output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{MODEL}_{task}.csv"
)
df.to_csv(output_path, index=False)

## Clasificación múltiple para Tara@5

Para analizar la estabilidad de las predicciones, se aplicará TARa@5 sobre un subconjunto estratificado del dataset. Cada ejemplo se evalúa cinco veces con el mismo modelo, reutilizando la primera predicción y generando cuatro ejecuciones adicionales, lo que permite medir el grado de consistencia de las salidas.

In [11]:
print(f"Iniciando TARa@5 para dataset: {task} | Textos: {len(df_tara5)}")

output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{MODEL}_{task}_tara5.csv"
)

llamadas_restantes = RPD_LIMIT

for run in range(1, 5):
    col_name = f"pred_run{run+1}"

    if col_name in df_tara5.columns:
        continue

    if llamadas_restantes <= 0:
        print("Límite diario alcanzado. Continuar mañana.")
        break

    print(f"\n=== Ejecutando {col_name} | Llamadas disponibles: {llamadas_restantes} ===")

    df_run = df_tara5[["user_id", "texto"]].copy()

    # IMPORTANTE: pasar límite restante
    df_run, usadas = classify_with_daily_limit(df_run, task, rpd_limit=llamadas_restantes, run_id=run+1)

    llamadas_restantes -= usadas

    df_tara5[col_name] = df_run["pred"]

    df_tara5.to_csv(output_path, index=False)

    print(f"{col_name} completado | llamadas usadas: {usadas} | restantes: {llamadas_restantes}")

print("\nTARa@5 completado (o en progreso si hay límite diario)")

Iniciando TARa@5 para dataset: anx | Textos: 100

=== Ejecutando pred_run2 | Llamadas disponibles: 240 ===
🆕 Sin checkpoint previo. Empezando desde cero.
📋 Textos pendientes: 100 | Límite de hoy: 240
📅 Días restantes estimados: 1


Progreso de hoy: 100%|██████████| 100/100 [04:48<00:00,  2.89s/it, bloqueados=2, hoy=100]



💾 Checkpoint guardado. Total procesados: 98/100
pred_run2 completado | llamadas usadas: 100 | restantes: 140

=== Ejecutando pred_run3 | Llamadas disponibles: 140 ===
🆕 Sin checkpoint previo. Empezando desde cero.
📋 Textos pendientes: 100 | Límite de hoy: 140
📅 Días restantes estimados: 1


Progreso de hoy: 100%|██████████| 100/100 [04:48<00:00,  2.89s/it, bloqueados=2, hoy=100]



💾 Checkpoint guardado. Total procesados: 98/100
pred_run3 completado | llamadas usadas: 100 | restantes: 40

=== Ejecutando pred_run4 | Llamadas disponibles: 40 ===
🆕 Sin checkpoint previo. Empezando desde cero.
📋 Textos pendientes: 100 | Límite de hoy: 40
📅 Días restantes estimados: 3


Progreso de hoy:  35%|███▌      | 14/40 [00:34<01:03,  2.46s/it, bloqueados=0, hoy=14]

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_requests_per_model_per_day, limit: 250, model: gemini-3.1-pro\nPlease retry in 3h45m32.695956039s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_requests_per_model_per_day', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel', 'quotaDimensions': {'model': 'gemini-3.1-pro', 'location': 'global'}, 'quota

In [12]:
checkpoint_manual = checkpoint_dir + f"tara5_{MODEL.replace('/', '_')}_{task}_run{run+1}.csv"
df_run.to_csv(checkpoint_manual, index=False)
print(f"💾 Guardado manual de seguridad: {checkpoint_manual}")

💾 Guardado manual de seguridad: /content/drive/MyDrive/tfg/corpusMentalRiskES/checkpoints/tara5_gemini-3.1-pro-preview_anx_run4.csv
